# Translator Training on Colab GPU

Einfacher Start fuer den ersten GPU-Run: GPU in Colab aktivieren, in den Konfig-Zellen nur den Datensatzpfad anpassen, dann `Run all`.

Ablauf: erst Preflight mit `check_dataset(...)`, danach Training mit `Trainer(...).train(...)`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/miwehle/Translator2.git'
REPO_BRANCH = 'integration/p-to-t'
REPO_DIR = Path('/content/Translator2')

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Colab; skipping Drive mount.')

if REPO_DIR.exists():
    print(f'Reusing existing checkout at {REPO_DIR}')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
else:
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
subprocess.run(
    ['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH],
    check=True,
)
os.chdir(REPO_DIR)
%pip install -q torch datasets

SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'Working directory: {Path.cwd()}')
print(f'Git branch: {REPO_BRANCH}')

In [ ]:
from pathlib import Path

DATASET_PATH = Path('/content/drive/MyDrive/translator_data/europarl.preprocessed')
RUN_NAME = 'run1'
MAX_EXAMPLES = None

TRAIN_KWARGS = {
    'epochs': 3,
    'num_workers': 2,
}

DATASET_PATH

In [ ]:
import sys

import torch

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from translator.data_prod import load_arrow_records
from translator.train_prod import Trainer, TrainerConfig, check_dataset

print(f"device={'cuda' if torch.cuda.is_available() else 'cpu'}")

if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATASET_PATH}')

ds = load_arrow_records(DATASET_PATH)

RUN_DIR = Path('/content/drive/MyDrive/translator_runs') / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

check_result = check_dataset(
    ds,
    max_examples=MAX_EXAMPLES,
)
check_result

In [ ]:
trainer_cfg = TrainerConfig(
    src_pad_idx=check_result['src_pad_idx'],
    tgt_pad_idx=check_result['tgt_pad_idx'],
    tgt_sos_idx=check_result['tgt_sos_idx'],
    src_vocab_size=check_result['src_vocab_size'],
    tgt_vocab_size=check_result['tgt_vocab_size'],
    num_examples=check_result['num_examples'],
    max_examples=MAX_EXAMPLES,
)

summary = Trainer(trainer_cfg).train(
    ds,
    checkpoint_path=RUN_DIR / 'model.pt',
    summary_path=RUN_DIR / 'summary.json',
    **TRAIN_KWARGS,
)
summary